# Datos Masivos — McCain: ¿dónde se vende la papa industrial en México?

**Entregable 2 en Databricks** · El mismo pipeline del repositorio, con el data lake en Unity Catalog

Este notebook vive en el repositorio de GitHub y lo abrimos en Databricks como **Git folder**.
No copiamos código aquí: llamamos a los mismos módulos de `src/` que corremos desde la terminal.
Lo único que cambia es **dónde guardamos los datos**: en lugar de `data/` en una computadora,
las capas viven en un Volume de Unity Catalog.

```
/Volumes/workspace/mccain_papa/datalake/
├── raw/          copia cruda de SIAP y SNIIM, sin modificar
├── interim/      esquema unificado, todo como texto (Parquet)
├── processed/    datos tipados y limpios (Parquet particionado)
└── reports/      resultados de la validación
```

Al final registramos la capa `processed` como tablas Delta en el catálogo
(`workspace.mccain_papa.*`), para consultarlas con SQL.

**Cómo correrlo:** conectar a *Serverless* y dar **Run all**.

## 0. Instalar librerías

Instalamos solo lo que no trae Databricks. **No** instalamos `requirements.txt` completo:
traería otra versión de `pyspark` encima de la que administra la plataforma.

In [ ]:
%pip install -q beautifulsoup4 pyyaml

In [ ]:
dbutils.library.restartPython()

## 1. Configurar el data lake

Creamos el esquema y el Volume si no existen, y le decimos al pipeline dónde guardar las capas
con variables de entorno. Las definimos **antes** de importar `src/`, porque `src/utils/rutas.py`
las lee al importarse.

In [ ]:
import os
import sys
from pathlib import Path

CATALOGO = "workspace"
ESQUEMA = "mccain_papa"
VOLUMEN = "datalake"
LAGO = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUMEN}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA} "
          "COMMENT 'Datos Masivos 2026 - papa industrial en Mexico (SIAP y SNIIM)'")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN} "
          "COMMENT 'Data lake del proyecto: raw, interim, processed y reports'")

os.environ["DATOS_MASIVOS_DATA"] = LAGO
os.environ["DATOS_MASIVOS_REPORTS"] = f"{LAGO}/reports"


def encontrar_raiz():
    """Buscamos hacia arriba la carpeta del repo (la que contiene src/)."""
    for candidata in [Path.cwd(), *Path.cwd().parents]:
        if (candidata / "src" / "processing" / "build_dataset.py").exists():
            return candidata
    raise RuntimeError("Abre este notebook desde el Git folder del repositorio.")


RAIZ = encontrar_raiz()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
os.chdir(RAIZ)

from src.utils import rutas

print("Repositorio :", RAIZ)
print("Data lake   :", rutas.DATA)
print("En Databricks:", rutas.EN_DATABRICKS)

## 2. Adquisición

Primero revisamos si Databricks alcanza los portales del gobierno. Según lo que encontremos
elegimos el modo:

| Modo | Cuándo | Qué hace |
|---|---|---|
| `completo` | Los portales responden | Descarga SIAP y SNIIM al Volume (unos 4 minutos) |
| `zip` | No responden, pero subimos `raw.zip` al Volume | Descomprime en `raw/` la descarga que hicimos en local |
| `muestra` | Ninguna de las dos | Reconstruye `raw/` desde las copias del E1 (SNIIM solo enero–marzo) |

Para el modo `zip`: en local, dentro de `data/`, comprimimos la carpeta `raw` como `raw.zip` y la
subimos en **Catalog → workspace → mccain_papa → datalake → Upload to this volume**.

In [ ]:
import requests

PORTALES = {
    "SIAP": "https://nube.agricultura.gob.mx/datosAbiertos/Agricola.php",
    "SNIIM": "http://www.economia-sniim.gob.mx",
}


def responde(url):
    try:
        return requests.get(url, timeout=15).status_code < 500
    except requests.RequestException:
        return False


alcance = {fuente: responde(url) for fuente, url in PORTALES.items()}
ZIP = Path(LAGO) / "raw.zip"

if all(alcance.values()):
    MODO = "completo"
elif ZIP.exists():
    MODO = "zip"
else:
    MODO = "muestra"

print("Portales:", alcance)
print("Modo elegido:", MODO)

In [ ]:
import zipfile

from src.ingestion import download_data

if MODO == "completo":
    codigo = download_data.main([])
elif MODO == "zip":
    with zipfile.ZipFile(ZIP) as archivo:
        # El zip trae la carpeta raw/ completa; la extraemos dentro del lago.
        archivo.extractall(LAGO)
    print("raw.zip descomprimido en", rutas.RAW)
    codigo = 0
else:
    codigo = download_data.main(["--muestra"])

print("Código de salida:", codigo)

In [ ]:
for carpeta in sorted(p for p in rutas.RAW.iterdir() if p.is_dir()):
    archivos = [a for a in carpeta.rglob("*") if a.is_file()]
    mb = sum(a.stat().st_size for a in archivos) / 1024**2
    print(f"raw/{carpeta.name:6s} | {len(archivos):3d} archivos | {mb:6.1f} MB")

## 3. Procesamiento con Spark

`build_dataset` lee `raw/`, escribe `interim/` y luego `processed/`, todo dentro del Volume.
En serverless no hay RDDs ni `cache()`; por eso el parseo del SNIIM usa la API de DataFrame
(`read.text(wholetext=True)` + UDF) y el mismo código corre aquí y en local.

In [ ]:
import json

from src.processing import build_dataset

codigo = build_dataset.main()
print("Código de salida:", codigo)

resumen = json.loads((rutas.PROCESSED / "resumen_build.json").read_text())
resumen["etapas"]["interim_a_processed"]

## 4. Validación

In [ ]:
import pandas as pd

from src.validation import validate_dataset

codigo = validate_dataset.main()
print("Código de salida:", codigo)

validacion = pd.read_csv(rutas.TABLAS / "validacion.csv")
print(validacion.estado.value_counts().to_dict())
display(validacion[validacion.estado != "OK"])

## 5. El data lake

Así quedaron las capas dentro del Volume.

In [ ]:
def tamano_mb(carpeta):
    return sum(a.stat().st_size for a in carpeta.rglob("*") if a.is_file()) / 1024**2


for capa in ["raw", "interim", "processed", "reports"]:
    ruta = Path(LAGO) / capa
    print(f"{capa + '/':12s} {tamano_mb(ruta):7.1f} MB")
    for hijo in sorted(p for p in ruta.iterdir() if p.is_dir()):
        particiones = sorted(p.name for p in hijo.iterdir() if p.is_dir())
        detalle = f"  particiones: {particiones[0]} … {particiones[-1]}" if particiones else ""
        print(f"   └── {hijo.name}/{detalle}")

## 6. Tablas en el catálogo

Registramos la capa `processed` como tablas Delta en `workspace.mccain_papa`. Así cualquiera con
acceso al workspace puede consultarlas con SQL sin saber dónde están los archivos.

In [ ]:
TABLAS_CATALOGO = {
    "siap_produccion": "Produccion de papa por municipio, anio, ciclo y modalidad (SIAP)",
    "sniim_precios": "Cotizaciones diarias de papa en mercados mayoristas (SNIIM)",
    "canalizacion": "Indice de canalizacion por estado y anio: % produccion - % cotizaciones",
}

for tabla, descripcion in TABLAS_CATALOGO.items():
    nombre = f"{CATALOGO}.{ESQUEMA}.{tabla}"
    (
        spark.read.parquet(str(rutas.PROCESSED / tabla))
        .write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nombre)
    )
    spark.sql(f"COMMENT ON TABLE {nombre} IS '{descripcion}'")
    print(f"{nombre:42s} {spark.table(nombre).count():>7,} filas")

## 7. Resultado: índice de canalización

In [ ]:
display(spark.sql(f"""
    SELECT ranking, estado,
           ROUND(produccion_ton)      AS produccion_ton,
           ROUND(pct_nacional, 1)     AS pct_nacional,
           ROUND(pct_mayoreo, 1)      AS pct_mayoreo,
           ROUND(canalizacion, 1)     AS canalizacion
    FROM {CATALOGO}.{ESQUEMA}.canalizacion
    WHERE anio = (SELECT MAX(anio) FROM {CATALOGO}.{ESQUEMA}.canalizacion)
    ORDER BY ranking
"""))

Positivo: el estado produce más de lo que aparece en el mercado spot, su papa sale por contrato.
Negativo: aparece más en mayoreo de lo que produce (redistribuye). En la tabla de arriba se puede
cambiar a gráfica de barras con el botón **+ → Visualization**.

## 8. Partition pruning sobre el Volume

`sniim_precios` está particionada por `anio` y `mes`. Al filtrar un mes, Spark solo lee esa
carpeta: buscamos `PartitionFilters` en el plan.

In [ ]:
from pyspark.sql import functions as F

enero = (
    spark.read.parquet(str(rutas.PROCESSED / "sniim_precios"))
    .filter((F.col("anio") == 2025) & (F.col("mes") == 1))
)
enero.explain(mode="formatted")

## 9. Conclusión

1. **El repositorio de GitHub es la fuente del código.** Databricks lo abre como Git folder; no
   hay copias del código en el workspace.
2. **El data lake vive en Unity Catalog.** `raw`, `interim`, `processed` y `reports` están en el
   Volume `workspace.mccain_papa.datalake`.
3. **El mismo pipeline corre local y en Databricks.** Solo cambian las variables de entorno.
4. **La capa final queda como tablas Delta** en `workspace.mccain_papa`, listas para el análisis
   del examen final.